In [1]:
!pip install -q ultralytics
import os, glob, json, time, zipfile, shutil, yaml
import numpy as np, pandas as pd
import torch
WORK = "/kaggle/working"
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.7 MB/s eta 0:00:00
torch: 2.10.0+cu128 | CUDA: True


In [3]:
import glob, os
for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
    if os.path.isfile(p):
        print(round(os.path.getsize(p)/1e6, 1), "MB", p)

3.5 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/__notebook__.ipynb
2.0 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/__output__.json
0.5 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/__results__.html
0.0 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/__results___files/__results___12_1.png
0.9 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/__results___files/__results___13_0.png
0.8 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/__results___files/__results___13_1.png
0.8 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/__results___files/__results___13_2.png
0.0 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/__results___files/__results___3_0.png
16892.1 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/_output_.zip
0.0 MB /kaggle/input/notebooks/payal1238gupta/ppe-2-0/custom.css
1.0 MB /kaggle/input/notebooks/payal1238gupta/ppe-3-0-v2/__notebook__.ipynb
1.6 MB /kaggle/input/notebooks/payal1238gupta/ppe-3-0-v2/__output__.json
0.5 MB /kaggle/input/notebooks/payal1

In [4]:
import zipfile, glob, os

P3ZIP = glob.glob("/kaggle/input/**/ppe-3-0-v2/_output_.zip", recursive=True)[0]

with zipfile.ZipFile(P3ZIP) as zf:
    pts = [m for m in zf.namelist() if m.endswith(".pt")
           and not any(b in m.lower() for b in ("smoke", "probe", "phase2", "last.pt"))]
    print("pt files in phase3 zip:", pts)
    def rank(m):
        n = os.path.basename(m).lower()
        return 0 if n == "phase3_best.pt" else (1 if "phase3" in m.lower() else 2)
    pts.sort(key=rank)
    assert pts, "phase3 zip me koi .pt nahi mila"
    zf.extract(pts[0], "/kaggle/working/ckpt")
    CKPT = f"/kaggle/working/ckpt/{pts[0]}"

from ultralytics import YOLO
model = YOLO(CKPT)
MODEL_NAMES = [model.names[i] for i in sorted(model.names)]
print("USING:", CKPT)
print("model classes:", MODEL_NAMES)
assert "Person" in MODEL_NAMES and "Gloves" in MODEL_NAMES, "wrong checkpoint!"

pt files in phase3 zip: ['yolov8n.pt', 'gloves_interim_train/gloves_det/weights/best.pt', 'phase3_best.pt', 'yolo11s.pt', 'yolo26n.pt', 'ppe_runs/phase3_custom_v2/weights/epoch30.pt', 'ppe_runs/phase3_custom_v2/weights/epoch0.pt', 'ppe_runs/phase3_custom_v2/weights/epoch10.pt', 'ppe_runs/phase3_custom_v2/weights/epoch65.pt', 'ppe_runs/phase3_custom_v2/weights/epoch45.pt', 'ppe_runs/phase3_custom_v2/weights/epoch25.pt', 'ppe_runs/phase3_custom_v2/weights/epoch35.pt', 'ppe_runs/phase3_custom_v2/weights/epoch20.pt', 'ppe_runs/phase3_custom_v2/weights/epoch40.pt', 'ppe_runs/phase3_custom_v2/weights/best.pt', 'ppe_runs/phase3_custom_v2/weights/epoch50.pt', 'ppe_runs/phase3_custom_v2/weights/epoch60.pt', 'ppe_runs/phase3_custom_v2/weights/epoch5.pt', 'ppe_runs/phase3_custom_v2/weights/epoch15.pt', 'ppe_runs/phase3_custom_v2/weights/epoch55.pt']
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Updat

In [5]:
import yaml, shutil

P2ZIP = glob.glob("/kaggle/input/**/ppe-2-0/_output_.zip", recursive=True)[0]
WORKZIP = P2ZIP

with zipfile.ZipFile(P2ZIP) as zf:
    inner = [m for m in zf.namelist() if m.endswith("ppe_unified.zip")]
    if inner:
        print("nested ppe_unified.zip mila — extract kar rahe:", inner[0])
        zf.extract(inner[0], "/kaggle/working/tmp")
        WORKZIP = f"/kaggle/working/tmp/{inner[0]}"

ROOT = None
with zipfile.ZipFile(WORKZIP) as zf:
    yamls = [m for m in zf.namelist() if m.endswith("data.yaml")]
    print("data.yaml candidates:", yamls)
    for m in yamls:
        y = yaml.safe_load(zf.read(m))
        names = y.get("names", [])
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if "Person" in names and "Gloves" in names:
            prefix = m[:-len("data.yaml")]
            members = [x for x in zf.namelist()
                       if x.startswith(prefix + "test/") or x == m]
            print(f"CHOSEN: {m} | classes ok | extracting {len(members)} test files…")
            zf.extractall("/kaggle/working/eval_ds", members=members)
            ROOT = f"/kaggle/working/eval_ds/{prefix}".rstrip("/")
            GT_NAMES = names
            break

assert ROOT, "koi data.yaml Person+Gloves wala nahi mila"
if os.path.exists("/kaggle/working/tmp"):
    shutil.rmtree("/kaggle/working/tmp", ignore_errors=True)   # disk bachao

TEST_IMG, TEST_LBL = f"{ROOT}/test/images", f"{ROOT}/test/labels"
imgs = sorted(glob.glob(f"{TEST_IMG}/*"))
print("ROOT:", ROOT, "| GT classes:", GT_NAMES, "| test images:", len(imgs))

nested ppe_unified.zip mila — extract kar rahe: ppe_unified.zip
data.yaml candidates: ['ppe_unified/data.yaml']
CHOSEN: ppe_unified/data.yaml | classes ok | extracting 3088 test files…
ROOT: /kaggle/working/eval_ds/ppe_unified | GT classes: ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle', 'Gloves', 'NO-Gloves'] | test images: 1542


In [6]:
NO_CLASSES = {"NO-Hardhat", "NO-Mask", "NO-Safety Vest", "NO-Gloves"}
PPE_CLASSES = {"Hardhat", "Mask", "Safety Vest", "Gloves"} | NO_CLASSES

gt = {}         
for img in imgs:
    stem = os.path.splitext(os.path.basename(img))[0]
    names_here = set()
    lf = f"{TEST_LBL}/{stem}.txt"
    if os.path.exists(lf):
        with open(lf) as f:
            for line in f:
                p = line.split()
                if p:
                    c = int(p[0])
                    if 0 <= c < len(GT_NAMES):
                        names_here.add(GT_NAMES[c])
    gt[stem] = names_here

n_person = sum(1 for s in gt.values() if "Person" in s)
n_ppe    = sum(1 for s in gt.values() if s & PPE_CLASSES)
n_both   = sum(1 for s in gt.values() if "Person" in s and (s & PPE_CLASSES))
n_viol   = sum(1 for s in gt.values() if s & NO_CLASSES)
print(f"images with Person: {n_person} | with PPE/NO-PPE: {n_ppe} | with BOTH: {n_both}")
print(f"GT violation images: {n_viol} / {len(gt)}")
PER_PERSON_OK = n_both >= 30   
print("per-person compliance computable:", PER_PERSON_OK)

images with Person: 154 | with PPE/NO-PPE: 1280 | with BOTH: 54
GT violation images: 464 / 1542
per-person compliance computable: True


In [7]:
t0 = time.time()
preds = {}      
BATCHSZ = 32
device = 0 if torch.cuda.is_available() else "cpu"
for i in range(0, len(imgs), BATCHSZ):
    batch = imgs[i:i+BATCHSZ]
    results = model.predict(batch, conf=0.05, imgsz=640, device=device, verbose=False)
    for img, r in zip(batch, results):
        stem = os.path.splitext(os.path.basename(img))[0]
        dets = []
        if r.boxes is not None:
            for c, cf in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist()):
                dets.append((model.names[int(c)], float(cf)))
        preds[stem] = dets
    if (i // BATCHSZ) % 10 == 0:
        print(f"  {i+len(batch)}/{len(imgs)}  ({time.time()-t0:.0f}s)")
print(f"inference done in {time.time()-t0:.0f}s")

  32/1542  (6s)
  352/1542  (10s)
  672/1542  (15s)
  992/1542  (25s)
  1312/1542  (38s)
inference done in 43s


In [8]:
def score(th):
    TP = TN = FP = FN = 0
    for stem, names_here in gt.items():
        gt_viol   = bool(names_here & NO_CLASSES)
        pred_viol = any(n in NO_CLASSES and cf >= th for n, cf in preds.get(stem, []))
        if   gt_viol and pred_viol:         TP += 1
        elif gt_viol and not pred_viol:     FN += 1
        elif not gt_viol and pred_viol:     FP += 1
        else:                               TN += 1
    acc  = (TP + TN) / max(1, TP + TN + FP + FN)
    prec = TP / max(1, TP + FP)
    rec  = TP / max(1, TP + FN)
    f1   = 2 * prec * rec / max(1e-9, prec + rec)
    return dict(threshold=th, accuracy=acc, precision=prec, recall=rec, f1=f1,
                TP=TP, TN=TN, FP=FP, FN=FN)

sweep = pd.DataFrame([score(t) for t in np.arange(0.10, 0.81, 0.05).round(2)])
print(sweep.round(4).to_string(index=False))

best = sweep.loc[sweep["accuracy"].idxmax()]
opr  = score(0.25)
print("\n================ HEADLINE ================")
print(f"Per-image violation-detection ACCURACY @ conf=0.25: {opr['accuracy']*100:.2f}%")
print(f"  (precision {opr['precision']:.3f} | recall {opr['recall']:.3f} | F1 {opr['f1']:.3f})")
print(f"Best accuracy over sweep: {best['accuracy']*100:.2f}% @ conf={best['threshold']}")
print("==========================================")

 threshold  accuracy  precision  recall     f1  TP   TN  FP  FN
      0.10    0.8969     0.7973  0.8815 0.8373 409  974 104  55
      0.15    0.9040     0.8173  0.8772 0.8462 407  987  91  57
      0.20    0.9099     0.8351  0.8728 0.8535 405  998  80  59
      0.25    0.9125     0.8478  0.8642 0.8559 401 1006  72  63
      0.30    0.9118     0.8596  0.8448 0.8522 392 1014  64  72
      0.35    0.9176     0.8856  0.8341 0.8590 387 1028  50  77
      0.40    0.9144     0.9049  0.7996 0.8490 371 1039  39  93
      0.45    0.9073     0.9303  0.7478 0.8292 347 1052  26 117
      0.50    0.8716     0.9290  0.6207 0.7442 288 1056  22 176
      0.55    0.8638     0.9472  0.5797 0.7193 269 1063  15 195
      0.60    0.8599     0.9733  0.5496 0.7025 255 1071   7 209
      0.65    0.8431     0.9744  0.4914 0.6533 228 1072   6 236
      0.70    0.8054     0.9824  0.3599 0.5268 167 1075   3 297
      0.75    0.7361     0.9672  0.1272 0.2248  59 1076   2 405
      0.80    0.7160     1.0000  0.0560 

In [9]:
if not PER_PERSON_OK:
    print("SKIPPED — per-person compliance accuracy is NOT computable on this dataset.")
    print(f"Reason: only {n_both} test images have Person AND PPE boxes annotated together")
    print("(the merged source datasets label either persons or PPE items, never both).")
    print("Per-image violation accuracy above is the honest headline figure.")
else:
    from itertools import product
    def load_boxes(stem):
        out = []
        lf = f"{TEST_LBL}/{stem}.txt"
        if os.path.exists(lf):
            with open(lf) as f:
                for line in f:
                    p = line.split()
                    if len(p) >= 5:
                        c = int(p[0]); x, y, w, h = map(float, p[1:5])
                        out.append((GT_NAMES[c], x, y, w, h))
        return out
    def inside(px, py, bx, by, bw, bh, pad=0.15):
        return (bx - bw/2 - pad <= px <= bx + bw/2 + pad) and (by - bh/2 - pad <= py <= by + bh/2 + pad)
    TPp = TNp = FPp = FNp = 0
    for stem in gt:
        boxes = load_boxes(stem)
        persons = [b for b in boxes if b[0] == "Person"]
        items   = [b for b in boxes if b[0] in PPE_CLASSES]
        pred_no = [(n, cf) for n, cf in preds.get(stem, []) if n in NO_CLASSES and cf >= 0.25]
        for _, px, py, pw, ph in persons:
            linked = [b for b in items if inside(b[1], b[2], px, py, pw, ph)]
            if not linked:
                continue
            gt_viol = any(b[0] in NO_CLASSES for b in linked)
            pd_viol = bool(pred_no)
            if   gt_viol and pd_viol:     TPp += 1
            elif gt_viol and not pd_viol: FNp += 1
            elif pd_viol:                 FPp += 1
            else:                         TNp += 1
    tot = TPp + TNp + FPp + FNp
    if tot:
        print(f"Per-person compliance accuracy: {(TPp+TNp)/tot*100:.2f}%  (n={tot})")
    else:
        print("No determinate person-PPE associations — falling back to per-image accuracy.")

Per-person compliance accuracy: 34.94%  (n=166)


In [10]:
results = {
    "phase": "3-accuracy",
    "checkpoint": CKPT,
    "test_images": len(imgs),
    "gt_violation_images": n_viol,
    "operating_point": opr,
    "best_over_sweep": best.to_dict(),
    "per_person_computable": bool(PER_PERSON_OK),
    "note": "Accuracy = per-image violation detection (any NO-* class), binary, (TP+TN)/total."
}
with open(f"{WORK}/phase3_accuracy.json", "w") as f:
    json.dump(results, f, indent=2, default=float)
sweep.to_csv(f"{WORK}/phase3_accuracy_sweep.csv", index=False)

print("Saved: phase3_accuracy.json, phase3_accuracy_sweep.csv\n")
print("---- paste-ready summary ----")
print(f"On the {len(imgs)}-image held-out test set, the Phase-3 detector achieves "
      f"{opr['accuracy']*100:.1f}% accuracy at the task of flagging images that contain a "
      f"PPE violation (precision {opr['precision']:.2f}, recall {opr['recall']:.2f}, "
      f"F1 {opr['f1']:.2f} at conf=0.25). Detection quality itself is reported as "
      f"mAP50 = 0.642 / mAP50-95 = 0.383, since box-level accuracy is undefined for detectors. "
      f"Per-person compliance accuracy is not computable on this dataset because the merged "
      f"sources never annotate Person and PPE boxes on the same image.")

Saved: phase3_accuracy.json, phase3_accuracy_sweep.csv

---- paste-ready summary ----
On the 1542-image held-out test set, the Phase-3 detector achieves 91.2% accuracy at the task of flagging images that contain a PPE violation (precision 0.85, recall 0.86, F1 0.86 at conf=0.25). Detection quality itself is reported as mAP50 = 0.642 / mAP50-95 = 0.383, since box-level accuracy is undefined for detectors. Per-person compliance accuracy is not computable on this dataset because the merged sources never annotate Person and PPE boxes on the same image.
